In [1]:
import os
import shutil
from pathlib import Path

DATASET_ROOT = "/kaggle/working/yolo_clove_dataset"
os.makedirs(f"{DATASET_ROOT}/images/train", exist_ok=True)
os.makedirs(f"{DATASET_ROOT}/labels/train", exist_ok=True)

print("🔄 Reorganizing dataset (more robust version)...")

# 1. Copy images
image_source = Path("/kaggle/input/datasets/patrickiitmz/clove-50-samples/sampled_50_per_class_224x224")
image_count = 0

for grade in ["Grade_1", "Grade_2", "Grade_3", "Grade_4"]:
    src = image_source / grade
    if src.exists():
        for img_file in src.glob("*.jpg"):
            dest = f"{DATASET_ROOT}/images/train/{img_file.name}"
            shutil.copy(img_file, dest)
            image_count += 1

print(f"✅ Copied {image_count} images")

# 2. Copy labels (search recursively — more flexible)
label_source = Path("/kaggle/input/datasets/patrickiitmz/clove-50-yolo-segmentation")
label_count = 0

for txt_file in label_source.rglob("*.txt"):
    dest = f"{DATASET_ROOT}/labels/train/{txt_file.name}"
    shutil.copy(txt_file, dest)
    label_count += 1

print(f"✅ Copied {label_count} label files")

# 3. Create correct data.yaml
yaml_content = """path: /kaggle/working/yolo_clove_dataset
train: images/train
val: images/train

names:
  0: Grade 1
  1: Grade 2
  2: Grade 3
  3: Grade 4
"""

with open(f"{DATASET_ROOT}/data.yaml", "w") as f:
    f.write(yaml_content)

print("✅ Created correct data.yaml")
print(f"\n🎉 Dataset ready at: {DATASET_ROOT}")

🔄 Reorganizing dataset (more robust version)...
✅ Copied 200 images
✅ Copied 200 label files
✅ Created correct data.yaml

🎉 Dataset ready at: /kaggle/working/yolo_clove_dataset


In [2]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 56.5 MB/s eta 0:00:00


In [3]:
from ultralytics import YOLO

model = YOLO("yolov8n-seg.pt")   # nano = best for mobile

results = model.train(
    data="/kaggle/working/yolo_clove_dataset/data.yaml",
    epochs=60,           # you can reduce to 40 if you want it faster
    imgsz=224,           # matches your image size
    batch=16,
    name="ubora_clove_yolov8",
    exist_ok=True,
    patience=20          # early stopping
)

# Export to TFLite (for your Flutter app)
model.export(format="tflite", imgsz=224, int8=True)
print("✅ Training finished + TFLite model exported!")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.38 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/yolo_clove_dataset/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=60, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7

E0000 00:00:1776368031.826967      22 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776368031.893903      22 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776368032.395381      22 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776368032.395426      22 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776368032.395429      22 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776368032.395432      22 computation_placer.cc:177] computation placer already registered. Please check linka

requirements: Ultralytics requirements ['sng4onnx>=1.0.1', 'onnx_graphsurgeon>=0.3.26', 'ai-edge-litert>=1.2.0', 'onnx2tf>=1.26.3,<1.29.0', 'onnxslim>=0.1.71', 'onnxruntime-gpu'] not found, attempting AutoUpdate...
Using Python 3.12.12 environment at: /usr
Resolved 18 packages in 2.30s
 Downloaded ai-edge-litert
 Downloaded onnxruntime-gpu
Prepared 7 packages in 2.43s
Installed 7 packages in 21ms
 + ai-edge-litert==2.1.4
 + backports-strenum==1.3.1
 + onnx-graphsurgeon==0.6.1
 + onnx2tf==1.28.8
 + onnxruntime-gpu==1.24.4
 + onnxslim==0.1.91
 + sng4onnx==2.0.1

requirements: AutoUpdate success ✅ 5.2s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect


TensorFlow SavedModel: starting export with tensorflow 2.19.0...
TensorFlow SavedModel: collecting INT8 calibration images from 'data=coco8-seg.yaml'

WARNING ⚠️ Dataset 'coco8-seg.yaml' images not found, missing path '/kaggle/working/datasets/coco8-seg/images/val'
Unzipping /kaggle/working/datasets/coco8

I0000 00:00:1776368059.778753      22 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13505 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1776368059.783899      22 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5
I0000 00:00:1776368063.335860      22 cuda_dnn.cc:529] Loaded cuDNN version 91002


Saved artifact at '/kaggle/working/runs/segment/ubora_clove_yolov8/weights/best_saved_model'. The following endpoints are available:

* Endpoint 'serving_default'
  inputs_0 (POSITIONAL_ONLY): TensorSpec(shape=(1, 224, 224, 3), dtype=tf.float32, name='images')
Output Type:
  List[TensorSpec(shape=(1, 40, 1029), dtype=tf.float32, name=None), TensorSpec(shape=(1, 56, 56, 32), dtype=tf.float32, name=None)]
Captures:
  134249788921360: TensorSpec(shape=(4, 2), dtype=tf.int32, name=None)
  134249788919824: TensorSpec(shape=(3, 3, 3, 16), dtype=tf.float32, name=None)
  134249788920592: TensorSpec(shape=(16,), dtype=tf.float32, name=None)
  134249788924816: TensorSpec(shape=(4, 2), dtype=tf.int32, name=None)
  134249788925968: TensorSpec(shape=(3, 3, 16, 32), dtype=tf.float32, name=None)
  134249788921168: TensorSpec(shape=(32,), dtype=tf.float32, name=None)
  134249788925584: TensorSpec(shape=(1, 1, 32, 32), dtype=tf.float32, name=None)
  134249788926160: TensorSpec(shape=(32,), dtype=tf.flo

I0000 00:00:1776368068.684269      22 devices.cc:67] Number of eligible GPUs (core count >= 8, compute capability >= 0.0): 2
I0000 00:00:1776368068.684511      22 single_machine.cc:374] Starting new session
I0000 00:00:1776368068.697990      22 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13505 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1776368068.699403      22 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5
W0000 00:00:1776368069.480434      22 tf_tfl_flatbuffer_helpers.cc:365] Ignored output_format.
W0000 00:00:1776368069.480470      22 tf_tfl_flatbuffer_helpers.cc:368] Ignored drop_control_dependency.
I0000 00:00:1776368070.348255      22 devices.cc:67] Number of eligible GPUs (core count >= 8, compute capability >= 0.0): 2
I0000 00:00:1776368070.348

W0000 00:00:1776368074.622818      22 tf_tfl_flatbuffer_helpers.cc:365] Ignored output_format.
W0000 00:00:1776368074.622850      22 tf_tfl_flatbuffer_helpers.cc:368] Ignored drop_control_dependency.
I0000 00:00:1776368074.666880      22 mlir_graph_optimization_pass.cc:425] MLIR V1 optimization pass is not enabled
fully_quantize: 0, inference_type: 6, input_inference_type: FLOAT32, output_inference_type: FLOAT32
W0000 00:00:1776368077.476400      22 tf_tfl_flatbuffer_helpers.cc:365] Ignored output_format.
W0000 00:00:1776368077.476445      22 tf_tfl_flatbuffer_helpers.cc:368] Ignored drop_control_dependency.
fully_quantize: 0, inference_type: 6, input_inference_type: INT8, output_inference_type: INT8
W0000 00:00:1776368080.316720      22 tf_tfl_flatbuffer_helpers.cc:365] Ignored output_format.
W0000 00:00:1776368080.316764      22 tf_tfl_flatbuffer_helpers.cc:368] Ignored drop_control_dependency.
W0000 00:00:1776368083.276055      22 tf_tfl_flatbuffer_helpers.cc:365] Ignored output_for

TensorFlow SavedModel: export success ✅ 55.8s, saved as '/kaggle/working/runs/segment/ubora_clove_yolov8/weights/best_saved_model' (41.4 MB)

TensorFlow Lite: starting export with tensorflow 2.19.0...
TensorFlow Lite: export success ✅ 0.0s, saved as '/kaggle/working/runs/segment/ubora_clove_yolov8/weights/best_saved_model/best_int8.tflite' (3.3 MB)

Export complete (56.0s)
Results saved to /kaggle/working/runs/segment/ubora_clove_yolov8/weights
Predict:         yolo predict task=segment model=/kaggle/working/runs/segment/ubora_clove_yolov8/weights/best_saved_model/best_int8.tflite imgsz=224 int8
Validate:        yolo val task=segment model=/kaggle/working/runs/segment/ubora_clove_yolov8/weights/best_saved_model/best_int8.tflite imgsz=224 data=/kaggle/working/yolo_clove_dataset/data.yaml int8 
Visualize:       https://netron.app
✅ Training finished + TFLite model exported!
